# Columnas desde DXF

Este notebook crea un DXF pequeno de prueba, extrae columnas no discretizadas y luego ejecuta la discretizacion.

In [6]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "dynaengine").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dynaengine import extract_columns_from_dxf, prepare_column_configs, process_column_config

In [7]:
MATERIALS = [
    {
        "material_name": "Material de poza",
        "unit_weight_kn_m3": 19,
        "shear_velocity": {"depth": [0, 5, 10, 15], "vs": [300, 350, 440, 550]},
        "shear_properties": {"c": 0, "phi": 34},
        "dynamic_model": {
            "model_type": "darendeli_2001",
            "sigma_vertical": 100,
            "soil_parameters": {"IP": 0.0, "OCR": 1.0, "k0": 0.7, "frequency": 1.0, "N": 10},
        },
    },
    {
        "material_name": "Material del dique",
        "unit_weight_kn_m3": 20,
        "shear_velocity": {"depth": [0, 8, 15, 20], "vs": [320, 380, 420, 550]},
        "shear_properties": {"c": 0, "phi": 34},
        "dynamic_model": {
            "model_type": "menq_2003",
            "sigma_vertical": 100,
            "soil_parameters": {"Cu": 18.0, "D50": 8.0, "k0": 0.7, "N": 10},
        },
    },
    {
        "material_name": "Grava arcillosa",
        "unit_weight_kn_m3": 19,
        "shear_velocity": {"depth": [0, 10, 20, 25], "vs": [200, 330, 540, 600]},
        "shear_properties": {"c": 0, "phi": 34},
        "dynamic_model": {
            "model_type": "wang_2021",
            "sigma_vertical": 100,
            "soil_parameters": {"soil_group": "Nonplastic silty sand group", "Cu": 2.0, "CF": 50.0, "e": 0.7, "D50": 1.0, "wc": 1.0, "k0": 0.7},
        },
    },
    {
        "material_name": "Grava arenosa",
        "unit_weight_kn_m3": 19,
        "shear_velocity": {"depth": [0, 24, 30, 40], "vs": [230, 300, 440, 700]},
        "shear_properties": {"c": 0, "phi": 34},
        "dynamic_model": {
            "model_type": "rollins_2020",
            "sigma_vertical": 100,
            "soil_parameters": {"Cu": 1.0, "k0": 1.0},
        },
    },
    {
        "material_name": "Grava pobremente gradada",
        "unit_weight_kn_m3": 19.0,
        "shear_velocity": {"depth": [0, 24, 30, 35], "vs": [230, 300, 440, 500]},
        "shear_properties": {"c": 0, "phi": 34},
        "dynamic_model": {
            "model_type": "ishibashi_1993",
            "sigma_vertical": 100,
            "soil_parameters": {"IP": 50.0, "k0": 0.5},
        },
    },
    {
        "material_name": "Estrato no identificado 1",
        "unit_weight_kn_m3": 19.0,
        "shear_velocity": {"depth": [0, 24, 30, 35], "vs": [230, 300, 440, 550]},
        "shear_properties": {"c": 0, "phi": 34},
        "dynamic_model": {
            "model_type": "rojas_2019",
            "sigma_vertical": 100,
            "soil_parameters": {"k0": 1.0},
        },
    },
]

In [8]:
dxf_path = ROOT / "examples" / "data" / "section_01.dxf"

extraction = extract_columns_from_dxf(dxf_path, x_positions=[250, 480])
extraction.material_names, extraction.unidentified_materials

(['Estrato no identificado 1',
  'Grava arcillosa',
  'Grava arenosa',
  'Grava pobremente gradada',
  'Material de poza'],
 ['Estrato no identificado 1'])

In [9]:
configs = prepare_column_configs(extraction.columns, MATERIALS, target_frequency_hz=25)
first_name = next(iter(configs))
result = process_column_config(configs[first_name], calibrate=False)
result.raw, result.discretized.head()

(   layer_id                    column_name             material_name  \
 0         1  section_01-column_1-failure_3          Material de poza   
 1         2  section_01-column_1-failure_3           Grava arcillosa   
 2         3  section_01-column_1-failure_3  Grava pobremente gradada   
 3         4  section_01-column_1-failure_3           Grava arcillosa   
 4         5  section_01-column_1-failure_3             Grava arenosa   
 
       top_m   bottom_m  center_depth_m  thickness_m  unit_weight_kn_m3  \
 0   0.00000    8.80462        4.402310      8.80462               19.0   
 1   8.80462   63.17001       35.987315     54.36539               19.0   
 2  63.17001   80.72154       71.945775     17.55153               19.0   
 3  80.72154   82.87277       81.797155      2.15123               19.0   
 4  82.87277  112.87257       97.872670     29.99980               19.0   
 
    shear_velocity_m_s  sigma_v_center_kpa       gmax_kpa      tau_kpa   k0  \
 0          319.737697       